# Global Air Quality Analysis Using PySpark

### Project Overview

This project uses PySpark to process, clean, and analyze global air quality data by combining WHO pollutant measurements (PM2.5, PM10, NO2) with country-level population data. The main goal is to calculate meaningful metrics such as a Pollution Index and a population-weighted Health Risk Index, enabling cross-country and temporal comparisons of air pollution and its impact on human health. The analysis focuses on identifying countries with the highest pollution exposure and population health risks, providing actionable insights for policymakers.

### Workflow Structure

The notebook demonstrates a complete PySpark data pipeline, starting from raw data ingestion to advanced cleaning, merging datasets, and deriving composite indices. Key steps include standardizing country names, handling missing or extreme values, imputing pollutant and population data, and calculating the Pollution Index and Health Risk Index. By combining pollutant intensity with population exposure, the project highlights countries like Pakistan, Bangladesh, and India as high-risk areas. The workflow also includes visualizations such as bubble charts, line charts, and stacked bar charts to communicate patterns and trends effectively. This approach showcases scalable data processing with Spark while providing clear, project-specific insights into global air pollution and health risks.

******************************

## Choice of Technology: Why Apache Spark

For this project, we chose **Apache Spark** to build and run our data pipeline. The decision was based on several practical and technical reasons:

### 1. Scalability and Distributed Processing
- Our dataset combines global air pollution data with population figures, giving us **tens of thousands of rows**.  
- Apache Spark can process data **across multiple machines**, making it fast and efficient.  
- Unlike tools like Pandas, Spark can **handle much larger datasets** without slowing down, which is useful for showing cloud-scale data processing.

### 2. Powerful DataFrame API and SQL
- Spark’s **DataFrame API** makes it easy to do complex tasks like:  
  - Filling or dropping missing values (`fill`, `dropna`, `coalesce`)  
  - Aggregations (`groupBy`, `agg`, `percentile_approx`)  
  - Conditional transformations (`withColumn`, `when`, `lit`)  
- We could also use **Spark SQL** for queries if needed, giving extra flexibility.

### 3. Performance Benefits
- Spark uses **lazy evaluation** and **query optimization**, so multiple steps run efficiently together.  
- It **parallelizes computations**, which reduces runtime compared to doing everything step by step in Python.

## 4. Why Not Other Technologies

We also considered other big data tools, but they were not the best fit for this project

- **Apache MapReduce** can process large datasets, but it is low-level and requires writing a lot of boilerplate code for each step. Implementing cleaning, transformations, and aggregations would be slow and cumbersome.

- **Hive** is useful for running SQL queries on structured data, but it is harder to perform step-by-step transformations, handle missing values, or calculate custom metrics like Pollution Index or Health Risk Index.

- **Storm** is designed for real-time stream processing, but our dataset is static. Using Storm would add unnecessary complexity without any benefit.

- **Apache Spark**, in contrast, combines high-level APIs with distributed processing. It supports batch operations, in-memory computation, SQL queries, and complex transformations efficiently. It also integrates well with Python for visualization, making it ideal for both processing and analyzing large datasets in a single workflow.

### 5. Easy Integration and Visualization
- Spark works well with **Python (PySpark)**, letting us run all steps in a **Jupyter notebook**.  
- This also makes it simple to visualize results and create reports, while heavy transformations are still done efficiently in Spark.

### 6. Conclusion
Using Apache Spark allows us to:  
- Process large datasets efficiently  
- Clean and transform data step by step  
- Calculate meaningful metrics like `pollution_index` and `health_risk_index`  
- Ensure scalability, speed, and reproducibility

*******************************************

# Step 1: Initialize Spark and Load Data

In [ ]:
from pyspark.sql import SparkSession

if SparkSession._instantiatedSession is not None:
    SparkSession._instantiatedSession.stop()

spark = SparkSession.builder \
    .appName("WHO-Population Integration") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

print("Spark session started")
print("App Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

In [ ]:
import os
import pandas as pd
import tempfile

who_xlsx_path = os.path.join("Datasets", "WHO Dataset.xlsx")
pop_csv_path  = os.path.join("Datasets", "POPULATION Dataset.csv")

# data is on the second sheet, not the default README sheet
who_pd = pd.read_excel(who_xlsx_path, sheet_name="AAP_2022_city_v9", engine="openpyxl")
_tmp = os.path.join(tempfile.gettempdir(), "who_tmp.csv")
who_pd.to_csv(_tmp, index=False, encoding="utf-8")
df_who = spark.read.csv(_tmp, header=True, inferSchema=True)

df_pop = spark.read.csv(pop_csv_path, header=True, inferSchema=True)

print("WHO Dataset Schema:")
df_who.printSchema()
df_who.show(5, truncate=False)

print("\nPopulation Dataset Schema:")
df_pop.printSchema()
df_pop.show(5, truncate=False)

### Step 1: Setting Up Spark and Loading Data

**Goal:**  
Here we start by setting up Spark and loading the two main datasets we will use: the WHO air quality data and the population data. This is the first step to make sure we can handle large datasets and perform all the necessary analysis.  

**What we did:**  
- **Start Spark:** Created a SparkSession called WHO-Population Integration to work with Spark DataFrames.  
- **Load files:** Read the CSV files using spark.read.csv with headers and automatic detection of column types.  
- **Check the data:** Looked at the schema and a few sample rows with printSchema and show to understand what the data looks like.  

**WHO Air Quality Data:**  
- **Size and columns:** 32,196 rows with columns like region, country, city, year, PM2.5, PM10, NO2, coverage percentages, reference, monitoring stations, version, and status.  
- **Notes:** Some pollutant values are missing for certain city-year pairs. This is normal and we will handle it later.  

**Population Data:**  
- **Size and columns:** 16,930 rows with country name, code, year, and population value.  
- **Notes:** Each row is for a country in a given year, covering multiple years.  

**Why we did this:**  
- Loading both datasets at the start lets us combine them later using country and year.  
- Checking the schema and sample rows makes sure Spark understood the data correctly and shows if there are any immediate issues.  
- This step sets the stage for cleaning, transforming, and merging the data in the next steps.

***********************************

# Step 2: Population Dataset Cleaning

In [ ]:
from pyspark.sql.functions import col, lower, trim, count, when, lit

for old, new in {"Country Name": "country_name", "Country Code": "country_code",
                 "Year": "year", "Value": "population"}.items():
    df_pop = df_pop.withColumnRenamed(old, new)

for c in ["country_name", "country_code"]:
    df_pop = df_pop.withColumn(c, lower(trim(col(c))))

df_pop.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_pop.columns
]).show(truncate=False)

df_pop.select(["year", "population"]).describe().show()
df_pop.select("country_name").distinct().show()
df_pop.select("year").distinct().show()

df_pop.printSchema()
df_pop.show(5, truncate=False)

### Step 2: Cleaning the Population Data

**Goal:**  
We cleaned and standardized the population dataset so it can be easily combined with the WHO air quality data later.  

**What we did:**  
- **Rename columns:** Changed all column names to lowercase: country_name, country_code, year, population. This makes it easier to work with them in Spark.  
- **Standardize strings:** Converted country names and codes to lowercase and removed extra spaces to avoid mismatches with WHO data.  
- **Check missing data:** Calculated how many values are missing in each column.  
- **Get basic stats:** Checked numbers for year and population to see the ranges and averages.  
- **Check distinct values:** Looked at the number of unique countries and years to understand coverage.  

**What we found:**  
- **Missing values:** None, all columns are complete.  
- **Numbers:**  
  - Years go from 1960 to 2023, with an average around 1991.  
  - Population ranges from 2,715 to over 8 billion, with an average around 216 million.  
- **Distinct countries and years:** Covers many countries over many years, so we can match with the WHO dataset.  
- **Sample rows:** Top rows show lowercase country names and codes, ready for merging.  

**Why this matters:**  
Cleaning and standardizing the population data prevents join errors with the WHO data and ensures accurate results in later analysis.

**************************

# Step 3: Merge WHO and Population Datasets

In [ ]:
from pyspark.sql.functions import col, create_map, lit, coalesce, lower, trim
from itertools import chain

df_who = df_who.withColumnRenamed("WHO Country Name", "who_country_name") \
               .withColumnRenamed("Measurement Year", "measurement_year")

df_who = df_who.withColumn("who_country_name", lower(trim(col("who_country_name"))))

country_name_map = {
    "united states of america": "united states",
    "czechia": "czech republic",
    "bolivia (plurinational state of)": "bolivia",
    "iran (islamic republic of)": "iran",
    "korea (democratic people's republic of)": "north korea",
    "korea (republic of)": "south korea",
    "russian federation": "russia",
    "syrian arab republic": "syria",
    "united republic of tanzania": "tanzania",
    "venezuela (bolivarian republic of)": "venezuela",
    "viet nam": "vietnam",
    "brunei darussalam": "brunei",
    "lao people's democratic republic": "laos",
    "moldova (republic of)": "moldova",
    "macedonia": "north macedonia",
    "micronesia (federated states of)": "micronesia",
    "palestine, state of": "palestine"
}

mapping_expr = create_map([lit(x) for x in chain(*country_name_map.items())])
df_who = df_who.withColumn(
    "who_country_name_std",
    coalesce(mapping_expr.getItem(col("who_country_name")), col("who_country_name"))
)

## Merging and Verification

In [ ]:
df_merged = df_who.join(
    df_pop,
    (df_who.who_country_name_std == df_pop.country_name) &
    (df_who.measurement_year == df_pop.year),
    how="left"
)

print("Merged dataset count:", df_merged.count())
df_merged.select(
    "who_country_name", "who_country_name_std", "country_name", "year", "population"
).show(10, truncate=False)

### Step 3: Combine WHO and Population Data

**Goal:**  
We merged the WHO air quality data with the population data to get a single dataset for analysis.  

**What we did:**  
1. **Standardize WHO country names**  
   - Country names in WHO data were inconsistent with mixed cases.  
   - Created a new column called who_country_name_std where names were converted to lowercase and mapped to fix mismatches (for example, United States of America → united states, Iran (Islamic Republic of) → iran).  
   - This ensures WHO country names match the population dataset.  

2. **Rename and clean columns**  
   - Columns with spaces like Measurement Year were renamed to measurement_year.  
   - Strings were trimmed and converted to lowercase to avoid errors from extra spaces or different cases.  

3. **Merge datasets**  
   - Did a left join of WHO data with population data using the standardized country name and year.  
   - Left join keeps all WHO rows even if population data is missing for some country-year combinations.  

4. **Check the merge**  
   - The merged dataset still has 32,196 rows, same as the original WHO data.  
   - Sample rows show that countries now have population values correctly aligned, e.g., Afghanistan 2019 has 37,856,121 people and Albania 2015 has 2,880,703.  
   - The column who_country_name_std helps track the fixed country names.  

**Why this matters:**  
Merging allows us to study pollution with population context. Accurate country-year alignment ensures that calculations like pollution per capita and health risk indexes are correct and consistent.

****************************************

# Step 4: Validate Merged Dataset

In [ ]:
from pyspark.sql.functions import count, when, lit

df_merged = df_merged.withColumnRenamed("PM2.5 (μg/m3)", "pm25") \
                     .withColumnRenamed("PM10 (μg/m3)", "pm10") \
                     .withColumnRenamed("NO2 (μg/m3)", "no2")

df_merged.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_merged.columns
]).show(truncate=False)

print("Duplicate rows:", df_merged.count() - df_merged.dropDuplicates().count())

df_merged.select(["pm25", "pm10", "no2", "population"]).describe().show()

df_merged.filter(
    (col("pm25") < 0) | (col("pm10") < 0) | (col("no2") < 0) | (col("population") <= 0)
).show(10, truncate=False)

### Step 4: Check and Validate the Merged Dataset

**Goal:**  
Make sure the merged dataset is clean, reliable, and ready for analysis.  

**Steps Taken:**  

1. **Rename pollutant columns**  
   - Columns like PM2.5 (μg/m3) were renamed to simpler names: pm25, pm10, no2.  
   - This makes calculations like pollution index and health risk easier.  

2. **Check for missing values**  
   - Looked at what percent of each column is missing.  
   - Observations:  
     - pm25, pm10, no2 have 30-53% missing values because some cities do not have full monitoring.  
     - Metadata columns like WHO Region, ISO3, country_name are almost complete.  
     - Status column is completely empty and can be ignored.  

3. **Check for duplicate rows**  
   - Found only 3 exact duplicate rows, which is very few and not a concern.  

4. **Descriptive statistics**  
   - Checked count, mean, standard deviation, min, and max for pm25, pm10, no2, and population.  
   - Observations:  
     - pm25 ranges from 0.01 to 191.9, pm10 from 1.04 to 540, showing high variation.  
     - Population ranges from about 35 thousand to 1.4 billion.  
     - High standard deviations show variability and possible outliers.  

5. **Look for impossible values**  
   - Checked for negative pollutant numbers or zero/negative population.  
   - No invalid values were found.  

**Key Takeaway:**  
The dataset is now standardized, cleaned, and validated. Missing data, duplicates, and outliers have been noted. We can confidently use this dataset for pollution indexes, health risk calculations, and visualizations.

***************************************

# Step 5: Advanced Cleaning and Preprocessing

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, when, count, lit, percentile_approx, coalesce

df_clean = df_merged.dropDuplicates()
df_clean = df_clean.dropna(subset=["who_country_name_std", "country_name", "year"])

pop_median_df = df_clean.groupBy("country_name").agg(
    percentile_approx("population", 0.5).alias("median_population")
)
df_clean = df_clean.join(pop_median_df, on="country_name", how="left") \
    .withColumn("population", coalesce(col("population"), col("median_population"))) \
    .drop("median_population")

df_clean = df_clean.filter((col("pm25") >= 0) | col("pm25").isNull()) \
                   .filter((col("pm10") >= 0) | col("pm10").isNull()) \
                   .filter((col("no2")  >= 0) | col("no2").isNull()) \
                   .filter(col("population") > 0)

for pollutant in ["pm25", "pm10", "no2"]:
    median_df = df_clean.groupBy("country_name").agg(
        percentile_approx(pollutant, 0.5).alias(f"{pollutant}_median")
    )
    df_clean = df_clean.join(median_df, on="country_name", how="left") \
                       .withColumn(pollutant, coalesce(col(pollutant), col(f"{pollutant}_median"))) \
                       .drop(f"{pollutant}_median")

## Validation and Checking

In [ ]:
for c in ["pm25_coverage", "pm10_coverage", "no2_coverage"]:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(
            c, when((col(c) < 0) | (col(c) > 100), None).otherwise(col(c))
        )

for c in ["pm25_level", "pm10_level", "no2_level"]:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(
            c, when(col(c).isin(["Low", "Medium", "High"]), col(c)).otherwise("Unknown")
        )

df_clean.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_clean.columns
]).show(truncate=False)

df_clean.printSchema()
print("Total rows after cleaning:", df_clean.count())
df_clean.show(10, truncate=False)
df_clean = df_clean.withColumnRenamed("population", "population_country_level")

### Step 5: Advanced Cleaning and Preprocessing

**Goal:**  
Make sure the dataset is clean, consistent, and ready for analysis.

**Steps Taken:**  

1. **Remove exact duplicates**  
   - Dropped duplicate rows to avoid bias in calculations.  
   - Result: 3 duplicates from the previous step were removed.

2. **Drop rows missing key identifiers**  
   - Any rows missing who_country_name_std, country_name, or year were removed.  
   - Ensures every row can be linked to a country and a year.

3. **Fill missing population values**  
   - Missing population numbers were replaced with the median population of that country.  
   - Now all countries have population values, which is important for per-capita calculations.

4. **Remove impossible values**  
   - Rows with negative pollutant values or zero/negative population were filtered out.  
   - Result: only valid measurements remain.

5. **Fill missing pollutant values (optional)**  
   - Missing pm25, pm10, or no2 values were filled with the country median.  
   - Missing percentages dropped to almost zero (pm25 ~0.37%, pm10 ~0.33%, no2 ~1.24%).

6. **Check temporal coverage columns**  
   - Coverage columns (0–100%) were verified and out-of-range values replaced with null.

7. **Validate pollutant level categories**  
   - pm25_level, pm10_level, no2_level were checked and invalid entries replaced with Unknown.

8. **Check missing values after cleaning**  
   - After cleaning, almost all important columns are fully populated.

9. **Final schema and rows**  
   - Total rows after cleaning: 30,247.  
   - Columns are standardized and ready for pollution and health risk calculations.

**Key Takeaway:**  
The dataset is now clean, consistent, and reliable. Duplicates, missing values, and extreme numbers have been handled. This prepares the data for calculating pollution indices, health risk scores, and making visualizations.

**Justification for using the median:**
We chose the median to fill missing values because pollutant levels and population numbers are not evenly distributed; some countries have extremely high values that would skew the mean. The median represents the central tendency more robustly, avoids bias from outliers, and ensures that imputed values do not distort the overall indices. Using the median rather than a random value or the mean makes the dataset statistically more reliable and defensible.

*********************************

# Step 6: Create Pollution Index and Health Risk Index

In [ ]:
from pyspark.sql.functions import col, when, coalesce, lit

df_final = df_clean

for pollutant, coverage in [("pm25", "PM25 temporal coverage (%)"),
                            ("pm10", "PM10 temporal coverage (%)"),
                            ("no2",  "NO2 temporal coverage (%)")]:
    if coverage in df_final.columns:
        df_final = df_final.withColumn(
            f"{pollutant}_adj",
            coalesce(col(pollutant), lit(0)) * coalesce(col(coverage) / 100, lit(0.5))
        )
    else:
        df_final = df_final.withColumn(f"{pollutant}_adj", coalesce(col(pollutant), lit(0)))

df_final = df_final.withColumn(
    "pollution_index",
    (
        coalesce(col("pm25_adj"), lit(0)) +
        coalesce(col("pm10_adj"), lit(0)) +
        coalesce(col("no2_adj"),  lit(0))
    ) / (
        when(col("pm25_adj").isNotNull(), 1).otherwise(0) +
        when(col("pm10_adj").isNotNull(), 1).otherwise(0) +
        when(col("no2_adj").isNotNull(),  1).otherwise(0)
    )
)

df_final = df_final.withColumn(
    "health_risk_index",
    (col("pollution_index") * col("population_country_level")) / 1_000_000_000
)

## Capping values, categorise and Display 10 rows 

In [ ]:
df_final = df_final.withColumn(
    "pollution_index",
    when(col("pollution_index") > 500, 500).otherwise(col("pollution_index"))
).withColumn(
    "health_risk_index",
    when(col("health_risk_index") > 100, 100).otherwise(col("health_risk_index"))
)

df_final = df_final.withColumn(
    "pm25_level",
    when(col("pm25") <= 12, "Low")
    .when((col("pm25") > 12) & (col("pm25") <= 35), "Medium")
    .otherwise("High")
).withColumn(
    "pm10_level",
    when(col("pm10") <= 20, "Low")
    .when((col("pm10") > 20) & (col("pm10") <= 50), "Medium")
    .otherwise("High")
).withColumn(
    "no2_level",
    when(col("no2") <= 40, "Low")
    .when((col("no2") > 40) & (col("no2") <= 80), "Medium")
    .otherwise("High")
)

df_final.select(
    "country_name", "year", "pm25", "pm10", "no2",
    "pollution_index", "health_risk_index",
    "pm25_level", "pm10_level", "no2_level",
    "population_country_level"
).show(10, truncate=False)

### Step 6: Create Pollution Index and Health Risk Index

**Goal:**  
Make two indices to summarize pollution and health risk so we can compare cities and countries easily.

**Steps Taken:**  

1. **Pollution Index**  
   - Calculated as the average of available pollutants (pm25, pm10, no2). Missing values are ignored.  
   - Formula: pollution_index = sum of available pollutants / number of available pollutants  
   - Result: Each row has one number showing overall pollution.  
   - Observation: Values range from very low (~0.3) to very high (~131.7) in polluted cities like Delhi.

2. **Population-Weighted Health Risk Index**  
   - Multiply pollution_index by country population to reflect total exposure.  
   - Formula: health_risk_index = (pollution_index * population_country_level) / 1,000,000,000  
   - Result: Shows total health burden. Big countries with high pollution have the highest values. Small countries have proportionally smaller values.

3. **Categorize Pollution Levels**  
   - Each pollutant is classified based on WHO thresholds:  
     - PM2.5: Low ≤12, Medium 12–35, High >35  
     - PM10: Low ≤20, Medium 20–50, High >50  
     - NO2: Low ≤40, Medium 40–80, High >80  
   - Result: New columns (`pm25_level`, `pm10_level`, `no2_level`) make it easy to see how severe the pollution is.

4. **Cap Extreme Values (Optional Check)**  
   - Pollution_index above 500 is capped to avoid extreme outliers.  
   - Health_risk_index stays reasonable because it is scaled by population.  
   - Observation: Most values remain realistic, confirming the data is clean.

5. **Final Verification**  
   - Checked sample rows to make sure indices and categories are correct.  
   - Example: North Macedonia 2019  
     - pm25=43.48 (High), pm10=37.88 (Medium), no2=21.18 (Low)  
     - pollution_index≈17.09, health_risk_index≈0.032  

**Key Takeaway:**  
Pollution_index and population-weighted health_risk_index let us rank countries by pollution and overall health risk. This makes comparisons fair across countries with different populations and helps produce clear and meaningful visualizations.

**********************

# Step 7 : inspecting and describing  final dataset

In [ ]:
from pyspark.sql.functions import col

df_final.show(10, truncate=False)
df_final.printSchema()

numeric_cols = ["pm25", "pm10", "no2", "pollution_index", "health_risk_index"]
df_final.select(numeric_cols).describe().show(truncate=False)

df_final.select([col(c).isNull().alias(c) for c in numeric_cols]).show(5)

for c in ["pm25_level", "pm10_level", "no2_level"]:
    if c in df_final.columns:
        print(f"\n{c}:")
        df_final.groupBy(c).count().show()

### Step 7: Final Dataset Checking and Description

**Goal:**  
Do a last check on the cleaned dataset to make sure everything is ready for analysis and visualizations.

**Steps and Observations:**  

1. **Dataset Overview**  
   - Contains country, city/locality, year, raw pollutants (pm25, pm10, no2), population, and the calculated indices (pollution_index, health_risk_index).  
   - Adjusted pollutant columns (pm25_adj, pm10_adj, no2_adj) are included.  
   - Pollutants are also grouped into Low, Medium, High based on WHO guidelines.

2. **Data Completeness**  
   - Pollution indices are fully populated, so comparisons across countries and cities are reliable.  
   - Some raw pollutant values had missing entries, but these were handled correctly when computing averages.

3. **Summary Statistics**  
   - Pollution_index ranges from 0.29 to 131.76, with an average of 16.3.  
   - Health_risk_index ranges from 0.00035 to 100, showing total population exposure differences.  
   - Most areas are Low or Medium for pollutant levels, with High being less common, especially for NO2.

4. **Purpose and Insights**  
   - Ensures calculations are correct and the dataset reflects real air quality patterns.  
   - Pollution levels differ a lot by country and pollutant, highlighting which areas are most at risk.

**Key Takeaway**  
   - Dataset is clean, consistent, and ready for visualizations and further analysis.  
   - Indices and pollutant levels accurately show air quality and total health risk, making it easy to rank countries or cities.

******************************